In [1]:
!pip install -q mlflow
!pip install -q git+https://github.com/RayenSansa03/aeroguard.git@main --force-reinstall --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
import shutil
import mlflow

DOSSIER = "/content/drive/MyDrive/aeroguard"
DOSSIER_MLFLOW_LOCAL = "/content/mlflow"
DOSSIER_MLFLOW_DRIVE = f"{DOSSIER}/mlflow"

if os.path.exists(DOSSIER_MLFLOW_DRIVE) and not os.path.exists(DOSSIER_MLFLOW_LOCAL):
    shutil.copytree(DOSSIER_MLFLOW_DRIVE, DOSSIER_MLFLOW_LOCAL)   # reprendre les séances précédentes
os.makedirs(DOSSIER_MLFLOW_LOCAL, exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{DOSSIER_MLFLOW_LOCAL}/mlflow.db")
print("MLflow", mlflow.__version__, "—", mlflow.get_tracking_uri())

Mounted at /content/drive
MLflow 3.16.1 — sqlite:////content/mlflow/mlflow.db


In [3]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from aeroguard.donnees_ml import preparer_xy, preparer_xy_famille

vols = pd.read_parquet(f"{DOSSIER}/data/processed/vols_ncmapss.parquet")
X_train, y_train = preparer_xy(vols, "train")
X_val, y_val = preparer_xy(vols, "val")

Xf_train, fam_train = preparer_xy_famille(vols, "train")
Xf_val, fam_val = preparer_xy_famille(vols, "val")
encodeur_familles = LabelEncoder().fit(fam_train)
yf_train = encodeur_familles.transform(fam_train)
yf_val = encodeur_familles.transform(fam_val)

In [6]:

import math

import mlflow
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

from aeroguard.donnees_ml import preparer_xy
from aeroguard.metier import evaluer_alerte
from aeroguard.modeles import REGLAGES_XGB, creer_xgboost, seuil_optimal

# --- 0. Vérifications : les cellules 1 à 3 ont-elles été exécutées ? ---
assert "vols" in globals(), "Exécute d'abord la cellule 3 (chargement des vols)."
assert mlflow.get_tracking_uri().startswith("sqlite"), "Exécute d'abord la cellule 2 (MLflow)."

NOM_EXPERIENCE = "aeroguard-alerte"
SEUIL_XGB_FIGE = 0.81   # seuil gelé en 3.6
FENETRE_CONFIRMATION = 3  # alerte confirmée sur 3 vols consécutifs
COLONNES_INFOS = ["moteur", "cycle", "RUL"]

mlflow.set_experiment(NOM_EXPERIENCE)


def scores_numeriques(scores):
    """Garde seulement les nombres valides (MLflow refuse le texte et les NaN)."""
    propres = {}
    for cle, valeur in scores.items():
        try:
            nombre = float(valeur)
        except (TypeError, ValueError):
            continue
        if not math.isnan(nombre):
            propres[cle] = nombre
    return propres


# --- 1. Données : train pour apprendre, validation pour mesurer ---
X_train, y_train = preparer_xy(vols, "train")
X_val, y_val = preparer_xy(vols, "val")
infos_val = vols.loc[X_val.index, COLONNES_INFOS]

print(f"Train : {X_train.shape}  |  Validation : {X_val.shape}")
print(f"Moteurs en validation : {infos_val['moteur'].nunique()}")  # attendu : 11

# --- 2. Run 1 : régression logistique ---
reglages_logistique = {
    "modele": "regression_logistique",
    "imputation": "median",
    "normalisation": "standard",
    "class_weight": "balanced",
    "max_iter": 1000,
    "fenetre_confirmation": FENETRE_CONFIRMATION,
}

with mlflow.start_run(run_name="logistique_alerte") as run:
    alerte_logistique = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(class_weight="balanced", max_iter=1000),
    )
    alerte_logistique.fit(X_train, y_train)

    probas_logistique = alerte_logistique.predict_proba(X_val)[:, 1]
    seuil_logistique, f1_macro_au_seuil = seuil_optimal(y_val, probas_logistique)

    scores_logistique = evaluer_alerte(
        y_val, probas_logistique, seuil_logistique, infos_val,
        k=FENETRE_CONFIRMATION, col_moteur="moteur",
    )

    mlflow.log_params({**reglages_logistique, "seuil": round(seuil_logistique, 2)})
    mlflow.log_metrics(scores_numeriques(scores_logistique))
    mlflow.set_tags({"tache": "alerte", "donnees": "N-CMAPSS", "lecon": "D3"})

    # ✅ Correction : name= et format cloudpickle (MLflow 3)
    mlflow.sklearn.log_model(
        alerte_logistique,
        name="model",
        serialization_format="cloudpickle",
    )
    run_id_logistique = run.info.run_id

print(f"✅ Logistique enregistrée — run {run_id_logistique[:8]}")

# --- 3. Run 2 : XGBoost sur GPU ---
with mlflow.start_run(run_name="xgboost_alerte") as run:
    alerte_xgboost = creer_xgboost(device="cuda", **REGLAGES_XGB)
    poids_train = compute_sample_weight("balanced", y_train)
    alerte_xgboost.fit(
        X_train, y_train,
        sample_weight=poids_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    probas_xgboost = alerte_xgboost.predict_proba(X_val)[:, 1]
    scores_xgboost = evaluer_alerte(
        y_val, probas_xgboost, SEUIL_XGB_FIGE, infos_val,
        k=FENETRE_CONFIRMATION, col_moteur="moteur",
    )

    mlflow.log_params({
        **REGLAGES_XGB,
        "modele": "xgboost",
        "device": "cuda",
        "seuil": SEUIL_XGB_FIGE,
        "fenetre_confirmation": FENETRE_CONFIRMATION,
        "meilleure_iteration": alerte_xgboost.best_iteration,
    })
    mlflow.log_metrics(scores_numeriques(scores_xgboost))
    mlflow.set_tags({"tache": "alerte", "donnees": "N-CMAPSS", "lecon": "D3"})

    # ✅ Correction : name= au lieu du 2e argument
    mlflow.xgboost.log_model(alerte_xgboost, name="model")
    run_id_xgboost = run.info.run_id

print(f"✅ XGBoost enregistré — run {run_id_xgboost[:8]}")

# --- 4. Comparaison rapide des deux runs ---
comparaison = mlflow.search_runs(
    experiment_names=[NOM_EXPERIENCE],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["start_time DESC"],
)
colonnes_affichees = [c for c in comparaison.columns
                      if c in ("tags.mlflow.runName", "params.seuil")
                      or c.startswith("metrics.")]
comparaison[colonnes_affichees].head(2)

Train : (3716, 126)  |  Validation : (819, 126)
Moteurs en validation : 11


2026/10/06 21:28:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Logistique enregistrée — run e615ef65


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [21:28:46] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


✅ XGBoost enregistré — run 6e07b0df


,metrics.avance_mediane,metrics.pr_auc,metrics.precision,metrics.moteurs_detectes,metrics.fp,metrics.moteurs,metrics.vn,metrics.f1,metrics.fn,metrics.avance_moyenne,metrics.vp,metrics.fausses_alarmes_total,metrics.moteurs_avec_fausse_alarme,metrics.rappel,metrics.f1_macro,params.seuil,tags.mlflow.runName
0,39.0,0.987461,0.971319,11.0,15.0,11.0,229.0,0.925319,67.0,42.000000,508.0,1.0,1.0,0.883478,0.886733,0.81,xgboost_alerte
1,44.0,0.986936,0.954545,11.0,25.0,11.0,219.0,0.933333,50.0,45.181818,525.0,2.0,2.0,0.913043,0.893567,0.37,logistique_alerte


In [18]:
from sklearn.metrics import f1_score
from aeroguard.evaluation import appliquer_seuil, metriques

with mlflow.start_run(run_name="balayage_max_depth"):
    for profondeur in range(1, 11):
        with mlflow.start_run(run_name=f"xgboost_depth_{profondeur}", nested=True):
            modele = creer_xgboost("cuda", max_depth=profondeur)
            modele.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
            modele.set_params(device="cpu")
            probas_val = modele.predict_proba(X_val)[:, 1]
            mlflow.log_param("max_depth", profondeur)
            mlflow.log_metrics({
                "f1_macro": f1_score(y_val, appliquer_seuil(probas_val, 0.5), average="macro"),
                "f1_train": f1_score(y_train, modele.predict(X_train), average="macro"),
                "pr_auc": metriques(y_val, appliquer_seuil(probas_val, 0.5), probas_val)["pr_auc"],
                "arbres_gardes": modele.best_iteration + 1,
            })
print("Balayage terminé : 10 runs enfants sous un run parent.")

Balayage terminé : 10 runs enfants sous un run parent.


In [8]:
from sklearn.metrics import accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from aeroguard.suivi import enregistrer_run

diagnostic_logistique = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                      LogisticRegression(max_iter=3000))
diagnostic_logistique.fit(Xf_train, yf_train,
                          logisticregression__sample_weight=compute_sample_weight("balanced", yf_train))
predictions_val = diagnostic_logistique.predict(Xf_val)

run_id = enregistrer_run(
    "diagnostic_logistique",
    reglages={"modele": "logistique", "classes": len(encodeur_familles.classes_),
              "ponderation": "balanced"},
    scores={"exactitude": accuracy_score(yf_val, predictions_val),
            "f1_macro": f1_score(yf_val, predictions_val, average="macro")},
    experience="aeroguard-diagnostic",
    etiquettes={"lecon": "3.3", "jeu": "validation",
                "limite": "famille = fichier NASA (voir 3.4 et 3.5)"},
)
print("Run de diagnostic enregistré :", run_id)

2026/10/06 21:29:20 INFO mlflow.tracking.fluent: Experiment with name 'aeroguard-diagnostic' does not exist. Creating a new experiment.


Run de diagnostic enregistré : a00fcc6fc7184c02aa8e66a57b7178ad


In [9]:
bilan_test_alerte = pd.read_csv(f"{DOSSIER}/results/v3_test_alerte.csv", index_col=0)
bilan_test_diagnostic = pd.read_csv(f"{DOSSIER}/results/v3_test_diagnostic.csv", index_col=0)

mlflow.set_experiment("aeroguard-evaluation-finale")
with mlflow.start_run(run_name="v3_test_final"):
    mlflow.log_params({"version": "v3", "moteurs_test": 39,
                       "alerte_retenue": "xgboost_seuil_0.81_k3"})
    for modele, ligne in bilan_test_alerte.iterrows():
        mlflow.log_metrics({f"alerte_{modele}_{cle}": float(valeur)
                            for cle, valeur in ligne.items()})
    for modele, ligne in bilan_test_diagnostic.iterrows():
        mlflow.log_metrics({f"diagnostic_{modele}_{cle}": float(valeur)
                            for cle, valeur in ligne.items()})
    for fichier in ["v3_test_alerte.csv", "v3_test_diagnostic.csv",
                    "v3_famille_non_vue.csv", "v3_vitesse_gpu.csv"]:
        mlflow.log_artifact(f"{DOSSIER}/results/{fichier}")
    mlflow.set_tags({"lecon": "3.6", "jeu": "test", "evaluation": "unique"})
print("Évaluation finale v3 enregistrée.")

2026/10/06 21:29:30 INFO mlflow.tracking.fluent: Experiment with name 'aeroguard-evaluation-finale' does not exist. Creating a new experiment.


Évaluation finale v3 enregistrée.


In [19]:
runs_alerte = mlflow.search_runs(experiment_names=["aeroguard-alerte"],
                                 filter_string="attributes.status = 'FINISHED'",
                                 order_by=["metrics.f1_macro DESC"])
colonnes = ["tags.mlflow.runName", "params.max_depth", "metrics.f1_macro",
            "metrics.pr_auc", "metrics.f1_train", "metrics.arbres_gardes"]
runs_alerte[[c for c in colonnes if c in runs_alerte.columns]].head(12).round(3)

,tags.mlflow.runName,params.max_depth,metrics.f1_macro,metrics.pr_auc,metrics.f1_train,metrics.arbres_gardes
0,logistique_alerte,None,0.894,0.987,NaN,NaN
1,xgboost_alerte,4,0.887,0.987,NaN,NaN
2,xgboost_depth_4,4,0.865,0.987,0.995,411.0
3,xgboost_depth_7,7,0.862,0.987,1.000,271.0
4,xgboost_depth_8,8,0.861,0.986,1.000,284.0
5,xgboost_depth_5,5,0.861,0.988,1.000,307.0
6,xgboost_depth_6,6,0.860,0.987,1.000,286.0
7,xgboost_depth_9,9,0.855,0.986,1.000,231.0
8,xgboost_depth_10,10,0.854,0.987,1.000,344.0
9,xgboost_depth_3,3,0.853,0.987,0.988,559.0


In [16]:
import subprocess
import time
import urllib.request

import mlflow
from google.colab import output

PORT_INTERFACE = 5000
FICHIER_JOURNAL = "/content/mlflow_serveur.log"
adresse_suivi = mlflow.get_tracking_uri()
print(f"Base MLflow : {adresse_suivi}")

# 1. Arrêter l'ancien serveur
subprocess.run(["pkill", "-f", "mlflow server"], check=False)
time.sleep(2)

# 2. Commande de base
commande = ["mlflow", "server",
            "--backend-store-uri", adresse_suivi,
            "--host", "127.0.0.1",
            "--port", str(PORT_INTERFACE)]

# 3. Sécurité MLflow 3 : autoriser le pont Colab
#    (serveur privé dans ta session Colab, accessible seulement par toi)
aide = subprocess.run(["mlflow", "server", "--help"],
                      capture_output=True, text=True).stdout
if "--allowed-hosts" in aide:
    commande += ["--allowed-hosts", "*"]
if "--cors-allowed-origins" in aide:
    commande += ["--cors-allowed-origins", "*"]        # ✅ nouveau : la correction
if "--x-frame-options" in aide:
    commande += ["--x-frame-options", "NONE"]
print("Commande :", " ".join(commande))

# 4. Lancer le serveur avec son journal
journal = open(FICHIER_JOURNAL, "w")
serveur_mlflow = subprocess.Popen(commande, stdout=journal, stderr=subprocess.STDOUT)

# 5. Attendre qu'il réponde (60 s maximum)
serveur_pret = False
for seconde in range(60):
    if serveur_mlflow.poll() is not None:
        break
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT_INTERFACE}", timeout=2)
        serveur_pret = True
        break
    except Exception:
        time.sleep(1)

# 6. Afficher l'interface
if serveur_pret:
    print(f"✅ Serveur prêt après {seconde} s")
    output.serve_kernel_port_as_iframe(PORT_INTERFACE, height=800)
else:
    print("❌ Le serveur ne répond pas. Journal :")
    journal.flush()
    print(open(FICHIER_JOURNAL).read()[-3000:])

Base MLflow : sqlite:////content/mlflow/mlflow.db
Commande : mlflow server --backend-store-uri sqlite:////content/mlflow/mlflow.db --host 127.0.0.1 --port 5000 --allowed-hosts * --cors-allowed-origins * --x-frame-options NONE
✅ Serveur prêt après 22 s


<IPython.core.display.Javascript object>

In [14]:
# Lire les erreurs du serveur MLflow
FICHIER_JOURNAL = "/content/mlflow_serveur.log"

lignes = open(FICHIER_JOURNAL).read().splitlines()
lignes_erreur = [i for i, ligne in enumerate(lignes)
                 if "Error" in ligne or "Exception" in ligne or "Traceback" in ligne]

if lignes_erreur:
    debut = max(0, lignes_erreur[-1] - 40)   # 40 lignes avant la dernière erreur
    print("\n".join(lignes[debut:lignes_erreur[-1] + 5]))
else:
    print("Aucune erreur dans le journal. Dernières lignes :")
    print("\n".join(lignes[-30:]))

Aucune erreur dans le journal. Dernières lignes :
2026/10/06 21:35:25 INFO:     127.0.0.1:55358 - "GET /static-files/static/js/3198.bb9ac57b.chunk.js HTTP/1.1" 200 OK
2026/10/06 21:35:25 INFO:     127.0.0.1:55378 - "GET /static-files/static/js/9848.1ab8e8e7.chunk.js HTTP/1.1" 200 OK
2026/10/06 21:35:25 INFO:     127.0.0.1:55382 - "GET /static-files/static/js/7182.19d2877a.chunk.js HTTP/1.1" 200 OK
2026/10/06 21:35:25 INFO:     127.0.0.1:55392 - "GET /static-files/static/js/4751.94bc6db7.chunk.js HTTP/1.1" 200 OK
2026/10/06 21:35:25 INFO:     127.0.0.1:55386 - "GET /static-files/static/js/8503.9d34794c.chunk.js HTTP/1.1" 200 OK
2026/10/06 21:35:25 INFO:     127.0.0.1:55404 - "GET /static-files/static/js/855.25446ffc.chunk.js HTTP/1.1" 200 OK
2026/10/06 21:35:25 INFO:     127.0.0.1:55406 - "GET /static-files/static/js/5702.205ea7af.chunk.js HTTP/1.1" 200 OK
2026/10/06 21:35:25 INFO:     127.0.0.1:55420 - "GET /static-files/static/js/9360.ea3236af.chunk.js HTTP/1.1" 200 OK
2026/10/06 21:3

In [20]:
import os
import shutil

# 1. Arrêter le serveur de l'interface (la base sqlite ne doit pas être utilisée pendant la copie)
if "serveur_mlflow" in globals():
    serveur_mlflow.terminate()

# 2. Trouver où MLflow range les artifacts (modèles, CSV)
dossiers_artifacts = set()
for experience in mlflow.search_experiments():
    emplacement = experience.artifact_location.replace("file://", "")
    if emplacement.startswith("/content") and not emplacement.startswith("/content/drive"):
        dossiers_artifacts.add(os.path.dirname(emplacement))
print("Dossiers d'artifacts :", dossiers_artifacts)

# 3. Copier la base et les artifacts sur Drive
shutil.copytree(DOSSIER_MLFLOW_LOCAL, DOSSIER_MLFLOW_DRIVE, dirs_exist_ok=True)
for dossier in dossiers_artifacts:
    if os.path.exists(dossier):
        destination = f"{DOSSIER}/{os.path.basename(dossier)}"
        shutil.copytree(dossier, destination, dirs_exist_ok=True)
        print(f"✅ {dossier} → {destination}")

print("✅ Base MLflow sauvegardée dans", DOSSIER_MLFLOW_DRIVE)

Dossiers d'artifacts : {'/content/mlruns'}
✅ /content/mlruns → /content/drive/MyDrive/aeroguard/mlruns
✅ Base MLflow sauvegardée dans /content/drive/MyDrive/aeroguard/mlflow
